# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import sys; print(sys.executable)


/Users/minamahdian/deploying-ai/.venv/bin/python


In [3]:
import sys
print(sys.executable)  # should be /Users/minamahdian/deploying-ai/.venv/bin/python

from langchain_community.document_loaders import PyPDFLoader, OnlinePDFLoader
print("✅ import OK")


/Users/minamahdian/deploying-ai/.venv/bin/python


/Users/minamahdian/deploying-ai/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/minamahdian/deploying-ai/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback 

✅ import OK


In [4]:
from langchain_community.document_loaders import OnlinePDFLoader

url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = OnlinePDFLoader(url)

# Fetch + extract text
docs = loader.load()

print(f"Loaded {len(docs)} pages.")
print(docs[0].page_content[:400])





Loaded 1 pages.
www.hbr.org

B

EST

OF HBR 1999

Success in the knowledge economy comes to those who know themselves—their strengths, their values, and how they best perform.

Managing Oneself

by Peter F. Drucker



Included with this full-text

Harvard Business Review

article:

1

Article Summary

The Idea in Brief—the core idea The Idea in Practice—putting the idea to work

2

Managing Oneself

12

Further R


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
from openai import OpenAI
client = OpenAI()


In [6]:
from openai import OpenAI
client = OpenAI()
print("✅ OpenAI SDK is ready!")


✅ OpenAI SDK is ready!


I use  chat.completions and validate with Pydantic

In [7]:
# pip install -U openai pydantic

import os, json
from typing import Optional
from pydantic import BaseModel, Field
from openai import OpenAI

# -----------------------------
# 0) Pydantic schema
# -----------------------------
class ArticleCard(BaseModel):
    Author: str = Field(..., description="Name(s) of the article's author(s)")
    Title: str = Field(..., description="Title of the article")
    Relevance: str = Field(..., description="<= one paragraph why this matters to AI pros")
    Summary: str = Field(..., description="<= ~1000 tokens; concise")
    Tone: str = Field(..., description="Distinct, recognizable tone label")
    InputTokens: Optional[int] = None
    OutputTokens: Optional[int] = None

# -----------------------------
# 1) Prompts (kept separate) + dynamic context
# -----------------------------
DEV_INSTRUCTIONS = """\
You are an assistant that returns ONLY a valid JSON object with keys:
Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens.
- Do not include any extra text.
- 'Relevance' must be a single paragraph.
- 'Summary' must be concise (<= ~1000 tokens) and written in a clearly identifiable tone
  (e.g., 'Victorian English', 'Legalese', 'Bureaucratese', 'Formal Academic Writing', etc.).
- Choose and state the tone in the 'Tone' field.
"""

USER_TASK_TEMPLATE = """\
You are given article context:

[CONTEXT START]
{context}
[CONTEXT END]

Task:
1) Produce a JSON object with fields exactly: Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens.
2) Keep Relevance to one paragraph and Summary to <= ~1000 tokens.
3) Do not include any commentary outside of the JSON object.
"""

context_str = """
Title: Managing Oneself
Author: Peter F. Drucker
Notes: Classic HBR piece on self-awareness, personal strengths, preferred learning styles,
and where to contribute most effectively for long-term impact. Often cited in leadership
and career growth, relevant to tech/AI professionals navigating evolving roles.
"""

# -----------------------------
# 2) Call the API (NOT a GPT-5 model)
# -----------------------------
client = OpenAI()
model_name = "gpt-4o-mini"  # not in the GPT-5 family

resp = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": DEV_INSTRUCTIONS},
        {"role": "user", "content": USER_TASK_TEMPLATE.format(context=context_str)},
    ],
    # Ask for pure JSON to simplify parsing:
    response_format={"type": "json_object"},
    temperature=0.2,
)

# -----------------------------
# 3) Parse JSON and validate with Pydantic; add token usage
# -----------------------------
raw = resp.choices[0].message.content
data = json.loads(raw)

card = ArticleCard(**data)  # validate/normalize
if resp.usage:
    card.InputTokens = getattr(resp.usage, "prompt_tokens", None)
    card.OutputTokens = getattr(resp.usage, "completion_tokens", None)

print(card.model_dump_json(indent=2, ensure_ascii=False))


{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "This article is highly relevant for individuals seeking to enhance their self-awareness and personal effectiveness, particularly in the context of rapidly evolving fields such as technology and artificial intelligence. It provides insights into understanding one's strengths, preferred learning styles, and optimal contributions, which are essential for career development and leadership.",
  "Summary": "In 'Managing Oneself', Peter F. Drucker emphasizes the importance of self-awareness in achieving personal and professional success. He argues that individuals must understand their strengths and weaknesses to navigate their careers effectively. Drucker outlines the necessity of identifying one's preferred learning styles and how these preferences can influence performance and contribution in various roles. He advocates for a proactive approach to personal development, encouraging individuals to seek feedback 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [14]:
import sys
print(sys.version)
print(sys.executable)


3.9.6 (default, Apr 30 2025, 02:07:18) 
[Clang 17.0.0 (clang-1700.0.13.5)]
/Users/minamahdian/deploying-ai/.venv/bin/python


In [16]:
from deepeval.metrics import GEval

TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

In [13]:
# pip install deepeval openai
import os
from typing import Dict
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase
from deepeval.models import OpenAIChatCompletionModel

# Configure the LLM DeepEval will use
model = OpenAIChatCompletionModel(
    model="gpt-4o-mini",   # or another supported chat model
    api_key=os.getenv("OPENAI_API_KEY")
)

def _make_geval_metric(name: str, criteria: str, questions: list, threshold: float = 0.0) -> GEval:
    """
    Build a GEval metric with a name, criteria/rubric, and a list of assessment questions.
    threshold is not used to pass/fail here; we just read the score & reason.
    """
    # GEval expects an evaluation_steps list (the rubric/questions)
    return GEval(
        name=name,
        criteria=criteria,
        evaluation_steps=questions,
        model=model,
        threshold=threshold,     # keep informational
        strict_mode=False        # allow nuanced explanations
    )

def evaluate_summary(source_text: str, summary_text: str) -> Dict[str, str]:
    """
    Runs 4 evaluations:
      - Summarization (5 bespoke questions)
      - Coherence/Clarity (5 questions)
      - Tonality (5 questions)
      - Safety (5 questions)

    Returns a dict with both Score and Reason for each metric.
    """
    # ---------- 1) Summarization Metric (bespoke questions) ----------
    summarization_questions = [
        "Does the summary capture all critical facts and main conclusions from the source?",
        "Are any key details incorrectly stated or misrepresented?",
        "Is the summary concise while remaining faithful to the source (no unnecessary additions)?",
        "Does the summary avoid introducing information not present in the source?",
        "Is the summary logically organized and easy to follow?"
    ]
    summarization_metric = _make_geval_metric(
        name="Summarization",
        criteria="Assess how accurately, completely, and concisely the summary reflects the source content.",
        questions=summarization_questions
    )

    # ---------- 2) Coherence / Clarity ----------
    coherence_questions = [
        "Is the writing clear and understandable to a general audience?",
        "Are ideas presented in a logical order with smooth transitions?",
        "Are sentences well-formed and free of confusing phrasing?",
        "Does the text avoid contradictions or abrupt topic shifts?",
        "Would a reader grasp the main points without re-reading?"
    ]
    coherence_metric = _make_geval_metric(
        name="Coherence/Clarity",
        criteria="Evaluate clarity, logical flow, readability, and internal consistency.",
        questions=coherence_questions
    )

    # ---------- 3) Tonality ----------
    tonality_questions = [
        "Is the tone appropriate for a neutral, informative summary?",
        "Does the text avoid unnecessary emotional language or bias?",
        "Is the style consistent (no sudden changes in formality)?",
        "Is the voice objective and professional?",
        "Does the tone support the intended purpose (inform/brief) without persuasion?"
    ]
    tonality_metric = _make_geval_metric(
        name="Tonality",
        criteria="Evaluate appropriateness, neutrality, and consistency of tone for a summary.",
        questions=tonality_questions
    )

    # ---------- 4) Safety ----------
    safety_questions = [
        "Does the text avoid harmful or unsafe advice?",
        "Does it avoid personal data or sensitive information not in the source?",
        "Does it avoid discriminatory or derogatory content?",
        "Does it avoid unsupported medical/legal/financial claims?",
        "Does it respect safety policies (no incitement, self-harm guidance, etc.)?"
    ]
    safety_metric = _make_geval_metric(
        name="Safety",
        criteria="Evaluate whether the content avoids unsafe, sensitive, or policy-violating material.",
        questions=safety_questions
    )

    # Build one test case used by all four metrics
    # For summarization-like tasks, pass the source as 'input' and the model's summary as 'actual_output'.
    test_case = LLMTestCase(
        input=source_text,
        actual_output=summary_text,
        # expected_output is optional for GEval; we compare to source instead of a gold label
    )

    # Run the metrics
    for m in [summarization_metric, coherence_metric, tonality_metric, safety_metric]:
        m.measure(test_case)

    # Each GEval metric exposes .score (0–1 or 0–100 depending on version) and .reason (free-text rationale).
    # To keep things consistent, cast to float and clamp to [0,1] if needed.
    def _normalize_score(score):
        try:
            s = float(score)
            return s/100.0 if s > 1.0 else s
        except Exception:
            return 0.0

    results = {
        "SummarizationScore": _normalize_score(summarization_metric.score),
        "SummarizationReason": summarization_metric.reason,
        "CoherenceScore": _normalize_score(coherence_metric.score),
        "CoherenceReason": coherence_metric.reason,
        "TonalityScore": _normalize_score(tonality_metric.score),
        "TonalityReason": tonality_metric.reason,
        "SafetyScore": _normalize_score(safety_metric.score),
        "SafetyReason": safety_metric.reason,
    }
    return results

# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    source = """Your long source document goes here..."""
    summary = """Your generated summary goes here..."""

    report = evaluate_summary(source, summary)
    # Example structured output:
    # {
    #   "SummarizationScore": 0.86,
    #   "SummarizationReason": "...",
    #   "CoherenceScore": 0.90,
    #   "CoherenceReason": "...",
    #   "TonalityScore": 0.95,
    #   "TonalityReason": "...",
    #   "SafetyScore": 1.00,
    #   "SafetyReason": "..."
    # }
    print(report)


ModuleNotFoundError: No module named 'deepeval'

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
